In [14]:
import polars as pl

(
    pl.scan_parquet(
        "s3://aind-scratch-data/dynamic-routing/cache/nwb_components/v0.0.268/consolidated/units.parquet"
    )
    .join(
        pl.scan_parquet(
            "s3://aind-scratch-data/dynamic-routing/cache/nwb_components/v0.0.268/consolidated/session.parquet"
        ).filter(
            pl.col('keywords').list.contains('ccf'),
            pl.col('keywords').list.contains('task'),
            pl.col('keywords').list.contains('production'),
            ~pl.col('keywords').list.contains('issues'),
            ~pl.col('keywords').list.contains('templeton'),
        ),
        on=['session_id'],
        how='semi',
    )
    .with_columns(
        (
            
        pl.col('activity_drift').le(0.2) & pl.col('amplitude_cutoff').le(0.1)
        & pl.col('presence_ratio').ge(0.7)
        & pl.col('isi_violations_ratio').le(0.5)
        & pl.col('decoder_label').ne('noise')
        & pl.col('firing_rate').ge(1)
        ).alias('is_good')
    )
    .group_by('session_id')
    .agg(
        pl.col('unit_id').filter('is_good').n_unique().alias('num_good_units'),
        pl.col('unit_id').filter(~pl.col('is_good')).n_unique().alias('num_bad_units'),
    )
    .sort('num_good_units')
    .collect()
)

session_id,num_good_units,num_bad_units
str,u32,u32
"""759434_2025-02-03""",78,910
"""626791_2022-08-16""",191,1371
"""714753_2024-07-02""",229,641
"""708016_2024-04-30""",273,349
"""699847_2024-04-15""",277,912
…,…,…
"""686176_2023-12-04""",1334,3285
"""715710_2024-07-18""",1362,3011
"""733780_2024-09-03""",1397,3046
